# 📊 PCB 缺陷检测训练总结

---

## 一、训练配置

| 配置项 | 值 |
|---|---|
| **模型** | YOLOv5s（预训练 `yolov5s.pt`） |
| **数据集** | PCB 缺陷数据集（6 类） |
| **图片尺寸** | 640×640 |
| **Batch Size** | 16 |
| **优化器** | SGD（lr=0.01，余弦退火） |
| **训练轮数** | 100 epochs |
| **总耗时** | ~3.1 小时 |
| **硬件** | RTX 3060 6GB |

---

## 二、最终指标

### 整体指标

| 指标 | 值 |
|---|---|
| **mAP@0.5** | **92.1%** |
| **mAP@0.5:0.95** | **63.5%** |
| **精确率 (Precision)** | **98.4%** |
| **召回率 (Recall)** | **90.6%** |

### 各类别详细指标

| 类别 | 中文名 | mAP@0.5 | 召回率 | 精确率 | 评估 |
|---|---|---|---|---|---|
| `missing_hole` | 缺孔 | **93.2%** | 92.6% | 99.8% | 🟢 优秀 |
| `mouse_bite` | 缺口 | **88.2%** | 87.9% | 95.3% | 🟢 良好 |
| `open_circuit` | 断路 | **95.9%** | 94.7% | 97.5% | 🟢 优秀 |
| `short` | 短路 | **92.5%** | 89.2% | 98.7% | 🟢 优秀 |
| `spur` | 毛刺 | **94.1%** | 92.3% | 99.2% | 🟢 优秀 |
| `spurious_copper` | 多余铜 | **88.9%** | 86.7% | 99.8% | 🟢 良好 |

> 所有 6 类缺陷 mAP@0.5 均超过 **88%**，整体精确率高达 **98.4%**，说明模型误检极少。

---

## 三、训练过程趋势

```mermaid
graph LR
    subgraph "Epoch 0 → 100"
        A["mAP@0.5<br/>~0% → 92.1%"] --> B["📈 大幅提升"]
        C["Loss<br/>box/obj/cls"] --> D["📉 持续下降"]
        E["Precision<br/>~0% → 98.4%"] --> F["📈 趋于稳定"]
        G["Recall<br/>~0% → 90.6%"] --> H["📈 趋于稳定"]
    end
```

- **损失曲线**：三类损失（box/obj/cls）在整个训练过程中持续下降，未出现明显过拟合
- **mAP 曲线**：mAP@0.5 从初始快速上升，约 60 轮后趋于平稳
- **精确率 vs 召回率**：精确率最终高于召回率，说明模型**宁可漏检也不误检**

---

## 四、推理性能

| 阶段 | 耗时/张 |
|---|---|
| **预处理** | 1.0ms |
| **推理** | 9.7ms |
| **NMS** | 1.3ms |
| **合计** | **~12ms**（约 83 FPS） |

> 在 RTX 3060 上达到 **实时检测** 水平（>30 FPS），适合生产线部署。

---

## 五、强项与弱项分析

### 🟢 强项（mAP > 93%）

| 类别 | 原因 |
|---|---|
| `open_circuit` (断路) | 特征明显，线条断开处对比清晰 |
| `spur` (毛刺) | 尖刺形状独特，容易区分 |
| `missing_hole` (缺孔) | 圆形孔洞缺失，形态规则 |
| `short` (短路) | 桥接特征显著 |

### 🟡 弱项（mAP < 90%）

| 类别 | 原因 | 改进方向 |
|---|---|---|
| `mouse_bite` (缺口) 88.2% | 缺口较小、形态多样、与背景对比度低 | 增加数据量、使用更大模型（yolov5m） |
| `spurious_copper` (多余铜) 88.9% | 铜渣形态不规则、分布随机 | 增加标注精度、使用 Mosaic 增强 |

---

## 六、最佳权重

| 文件 | 路径 | 大小 |
|---|---|---|
| **best.pt** | `yolov5-7.0/runs/train/exp4/weights/best.pt` | 14.5MB |
| last.pt | `yolov5-7.0/runs/train/exp4/weights/last.pt` | 14.5MB |

> `best.pt` 为验证集上表现最佳的权重，推荐用于推理部署。

---

## 七、推理结果验证

使用 `detect.py` 对验证集 720 张图片进行推理（置信度阈值 0.5）：

| 项目 | 结果 |
|---|---|
| 验证集图片 | 720 张 |
| 检测到的目标数 | 441 个 |
| 平均推理速度 | 9.7ms/张 |
| 输出结果位置 | `yolov5-7.0/runs/detect/exp/` |

---

## 八、后续优化建议

1. 🔄 **增加训练轮数**：当前曲线尚未完全收敛，可尝试 150~200 轮
2. 🔄 **使用更大模型**：`yolov5m` 或 `yolov5l` 可能提升 mAP 2~3%
3. 🔄 **数据增强**：针对 `mouse_bite` 和 `spurious_copper` 增加针对性样本
4. 🔄 **调整置信度阈值**：生产环境中可按需求调整 `conf-thres` 平衡精确率/召回率
5. 🔄 **模型导出**：可导出 ONNX 格式加速部署：`python export.py --weights best.pt --include onnx`


# 📐 评估指标详解 — 看懂训练输出

每次训练结束或验证时，YOLOv5 会输出一张表格和若干曲线图：

```
                 Class     Images  Instances          P          R      mAP50   mAP50-95
                   all        720        766      0.984      0.906      0.921      0.635
          missing_hole        720        108      0.998      0.926      0.932      0.698
            mouse_bite        720        116      0.953      0.879      0.882      0.601
```

下面逐一解释每个指标的含义和用途。

---

## 一、四个核心指标

### ① P — Precision（精确率）

$$P = \frac{TP}{TP + FP} = \frac{真正例}{真正例 + 假正例}$$

**人话：模型说"这是缺陷"的结果中，有多少是真的对了？**

```
                  预测为正          预测为负
实际为正     TP (正确检出)     FN (漏检)
实际为负     FP (误检)         TN (正确排除)
```

| 值 | 含义 |
|---|---|
| P = 98.4% | 模型检测出的缺陷中，98.4% 确实是真正的缺陷 |
| P 高 = **误检少** | 适合"宁可漏检不可误检"的场景 |

### ② R — Recall（召回率）

$$R = \frac{TP}{TP + FN} = \frac{真正例}{真正例 + 假负例}$$

**人话：所有真实的缺陷中，模型找出了多少？**

| 值 | 含义 |
|---|---|
| R = 90.6% | 所有真实缺陷中，模型找出了 90.6% |
| R 高 = **漏检少** | 适合"宁可误检不可漏检"的场景 |

### ③ Precision vs Recall 的权衡

```
高 P（低误检）←→ 高 R（低漏检）
   无法同时达到最高

    完美模型             现实权衡
    ┌─────┐            ┌─────┐
    │ P=1 │            │ P~0.98│
    │ R=1 │            │ R~0.91│
    └─────┘            └─────┘
    ↑ 不可能            ↑ 实际表现
```

- 提高置信度阈值 → P ↑ 但 R ↓（更保守）
- 降低置信度阈值 → R ↑ 但 P ↓（更激进）

> 我们的模型 P=98.4%, R=90.6%，属于**高精确率优先**，误检极少。

---

### ④ mAP — Mean Average Precision（平均精度均值）

这是目标检测中最**综合、最权威**的指标。

#### 构建过程

```mermaid
graph TD
    A["对每个类别<br/>按置信度排序"] --> B["计算不同阈值下的<br/>P 和 R"]
    B --> C["绘制 P-R 曲线"]
    C --> D["曲线下面积 = AP"]
    D --> E["所有类别 AP 平均 = mAP"]
```

#### mAP@0.5

- 设定 IoU 阈值 = 0.5（预测框和真实框重叠 > 50% 就算检对）
- 计算 AP，再对所有类别取平均
- 结果：**mAP@0.5 = 92.1%**
- ✅ 相对宽松的标准，反映模型"大致能检测到目标"的能力

#### mAP@0.5:0.95（严格标准）

- 对 IoU 阈值从 0.5 到 0.95，**每步 0.05** 分别计算 mAP，再取平均
- 结果：**mAP@0.5:0.95 = 63.5%**
- ✅ COCO 竞赛官方指标，反映模型"框得有多准"的能力

> **两个指标一起看：**
> - mAP@0.5 **高** → 模型能检测到目标（定位大致正确）
> - mAP@0.5:0.95 **高** → 模型框得非常精准（定位精确）

---

## 二、各类指标对比

| 类别 | P | R | mAP@0.5 | mAP@0.5:0.95 | 差距(宽松→严格) |
|---|---|---|---|---|---|
| `open_circuit` | 97.5% | 94.7% | **95.9%** | 65.9% | ↘ 30% |
| `spur` | 99.2% | 92.3% | **94.1%** | 64.6% | ↘ 29.5% |
| `missing_hole` | 99.8% | 92.6% | **93.2%** | 69.8% | ↘ 23.4% |
| `short` | 98.7% | 89.2% | **92.5%** | 64.1% | ↘ 28.4% |
| `spurious_copper` | 99.8% | 86.7% | **88.9%** | 56.8% | ↘ 32.1% |
| `mouse_bite` | 95.3% | 87.9% | **88.2%** | 60.1% | ↘ 28.1% |

**分析：**
- `missing_hole` 的 mAP@0.5→0.95 下降最小（23.4%），说明**框得最准**
- `spurious_copper` 下降最大（32.1%），说明**框的精度不够**，需要更好的定位
- 所有类别的 P 都 > 95%，说明**误检极少**，这是最大的优势

---

## 三、训练过程中输出的曲线图

训练完成后 `runs/train/exp4/` 目录下生成的可视化文件：

### 3.1 `results.png` — 训练总览

包含 4 行 × 3 列的 12 张小图：

```mermaid
graph TD
    subgraph "results.png 内容"
        A["第1行: 损失<br/>box_loss / obj_loss / cls_loss"]
        B["第2行: 指标<br/>Precision / Recall / mAP@0.5"]
        C["第3行: 指标<br/>mAP@0.5:0.95 / val_box / val_obj"]
        D["第4行: 学习率<br/>lr0 / lr1 / lr2"]
    end
```

**怎么看：**
- 损失曲线 → **持续下降** ✅ 正常
- mAP 曲线 → **持续上升** ✅ 正常
- 如果损失下降但 mAP 不升 → 过拟合 ⚠️
- 如果损失震荡剧烈 → 学习率太大 ⚠️

### 3.2 `PR_curve.png` — P-R 曲线

```
完美模型: 曲线紧贴右上角（P=1, R=1）
差模型:   曲线靠近左下角

  P
  ↑  ┌─────────────────
  1  │ ╱╲    ← 各类别 P-R 曲线
     │╱  ╲
     ╱    ╲
    ╱      ╲
  0 └───────────────→ R
    0              1
```

- 曲线下方面积 = AP
- 蓝色粗线 = 所有类别平均

### 3.3 `confusion_matrix.png` — 混淆矩阵

```
      预测类别
    ┌──┬──┬──┬──┬──┬──┬──┐
    │  │MH│MB│OC│SH│SP│SC│↘ 对角线越亮越好
真 ├──┼──┼──┼──┼──┼──┼──┤
实 │MH│93│ 0│ 1│ 2│ 0│ 0│  → 缺孔正确率 93%
类 ├──┼──┼──┼──┼──┼──┼──┤
别 │MB│ 0│88│ 0│ 1│ 0│ 2│  → 缺口 88% 正确
  ⋮
```

- **对角线** → 正确分类（越亮越好）
- **非对角线** → 误分类（越暗越好）
- **最后一列** → 漏检的背景目标

### 3.4 `labels.jpg` — 标签分布

```
显示数据集中：
┌────────────────────────┐
│ ■  GT 框的位置分布      │
│ ■  GT 框的大小分布      │
│ ■ 各类别数量统计        │
│ ■ 中心点位置热力图      │
└────────────────────────┘
```

**作用：** 检查数据集是否存在偏差
- 如果中心点全集中在图像中央 → 数据采集有偏
- 如果某个类别样本极少 → 可能需要补充数据

### 3.5 `F1_curve.png` — F1 曲线

$$F1 = 2 \times \frac{P \times R}{P + R}$$

- F1 是 P 和 R 的**调和平均**，综合衡量两者
- 曲线最高点对应的置信度阈值就是最优阈值
- 可用该值调整 `--conf-thres`

---

## 四、实际训练结果解读

以我们的训练结果为例：

```
                 Class     Images  Instances          P          R      mAP50   mAP50-95
                   all        720        766      0.984      0.906      0.921      0.635
```

| 指标 | 值 | 评价 |
|---|---|---|
| Images | 720 | 验证集图片数 |
| Instances | 766 | 验证集总目标数 |
| P | 98.4% | 🟢 极高，误检极少 |
| R | 90.6% | 🟢 良好，漏检较低 |
| mAP@0.5 | 92.1% | 🟢 优秀，检测准确 |
| mAP@0.5:0.95 | 63.5% | 🟡 中等偏上，框的精度有提升空间 |

---

## 五、总结速查

```mermaid
graph TD
    subgraph "指标速查"
        A["P (精确率)<br/>检出的东西里有几个对的？"]
        B["R (召回率)<br/>所有缺陷找出了几个？"]
        C["mAP@0.5<br/>宽松标准下检测准不准？"]
        D["mAP@0.5:0.95<br/>严格标准下框得准不准？"]
    end
    
    A --> E["98.4% ✅ 几乎没错检"]
    B --> F["90.6% ✅ 大部分找到"]
    C --> G["92.1% ✅ 检测很准"]
    D --> H["63.5% 🟡 框位还可优化"]

    style E fill:#90EE90
    style F fill:#90EE90
    style G fill:#90EE90
    style H fill:#FFD700
```

> **一句话总结：** 我们的模型**找得准（P=98.4%）、找得多（R=90.6%），在宽松标准下表现优秀（mAP@0.5=92.1%），在严格定位标准下仍有提升空间（mAP@0.5:0.95=63.5%）。**
